# GroupDNA — WhatsApp Group Chat Decoded
**"Spotify Wrapped, but for your friend group."**

---
| Field | Details |
|---|---|
| **Project** | GroupDNA – Minor Project I |
| **Name** | Raghav |
| **Batch** | Data Science – June |
| **Date** | 29 June 2026 |

> **Constraints followed:** No pandas · No matplotlib · No regex · No collections.Counter · Pure Python fundamentals + NumPy only


## Feature 1 — The Chat Parser
Reads `hostel_bois.txt`, parses every line, handles all 5 edge cases (system messages, media omitted, deleted messages, multi-line, empty lines).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 1: THE CHAT PARSER
# Reads hostel_bois.txt and extracts timestamp, sender, text from each message.
# Handles: system messages, media omitted, deleted messages, multi-line, empty lines.
# Uses: open(), str.split(), list of dicts, conditionals, loops, functions
# ─────────────────────────────────────────────────────────────────────────────

def is_date_line(line):
    """Return True if a line starts with the WhatsApp date pattern DD/MM/YY."""
    p = line.strip()
    if len(p) < 8:
        return False
    return (p[0:2].isdigit() and p[2] == '/' and
            p[3:5].isdigit() and p[5] == '/' and
            p[6:8].isdigit())


def parse_chat(filepath):
    """
    Parse a WhatsApp exported .txt file into a list of message dictionaries.

    Each dict has keys:
        'timestamp' : str  e.g. '01/04/24, 01:17'
        'sender'    : str  e.g. 'Rahul'
        'text'      : str  the message body
        'is_media'  : bool True if '<Media omitted>'
        'is_deleted': bool True if 'This message was deleted'

    Returns (messages, skipped_system, skipped_media, skipped_deleted)
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    messages        = []
    skipped_system  = 0
    skipped_media   = 0
    skipped_deleted = 0

    for line in lines:
        line = line.strip()

        # Edge case e: empty lines — skip silently
        if not line:
            continue

        # Edge case d: multi-line continuation — no timestamp prefix
        if not is_date_line(line):
            if messages:
                messages[-1]['text'] += ' ' + line
            continue

        # Need at least one ' - ' to separate timestamp from content
        if ' - ' not in line:
            skipped_system += 1
            continue

        timestamp_part, rest = line.split(' - ', 1)

        # Edge case a: system messages — no ': ' means no sender:message structure
        if ': ' not in rest:
            skipped_system += 1
            continue

        sender, text = rest.split(': ', 1)

        # Additional system message guard: very long senders or phone-number senders
        if len(sender) > 30 or sender.startswith('+'):
            skipped_system += 1
            continue

        is_media   = (text.strip() == '<Media omitted>')
        is_deleted = (text.strip() == 'This message was deleted')

        # Edge case b: media omitted — count separately, exclude from word freq
        if is_media:
            skipped_media += 1

        # Edge case c: deleted messages — count separately
        if is_deleted:
            skipped_deleted += 1

        messages.append({
            'timestamp' : timestamp_part.strip(),
            'sender'    : sender.strip(),
            'text'      : text.strip(),
            'is_media'  : is_media,
            'is_deleted': is_deleted
        })

    return messages, skipped_system, skipped_media, skipped_deleted


# ── Run the parser ────────────────────────────────────────────────────────────
# In Google Colab: upload hostel_bois.txt via the sidebar, then use this path:
CHAT_FILE = 'hostel_bois.txt'           # change to '/content/hostel_bois.txt' in Colab

messages, sys_skip, media_skip, del_skip = parse_chat(CHAT_FILE)

real_messages = [m for m in messages if not m['is_media'] and not m['is_deleted']]

participants  = sorted(set(m['sender'] for m in messages))

print(f"Successfully parsed {len(messages)} messages from {len(participants)} participants.")
print(f"Real messages   : {len(real_messages)}")
print(f"Media omitted   : {media_skip}")
print(f"Deleted messages: {del_skip}")
print(f"System messages : {sys_skip}")
print(f"\nParticipants: {', '.join(participants)}")
print(f"\nFirst message: {messages[0]}")
print(f"Last  message: {messages[-1]}")


## Feature 2 — Group Overview
Headline statistics: total messages, date range, participants, per-person message count.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 2: GROUP OVERVIEW
# Computes headline statistics and per-person message counts.
# Uses: dicts, sorted() with key=, f-strings, arithmetic, sets
# ─────────────────────────────────────────────────────────────────────────────
from datetime import datetime

PARTICIPANTS = ['Rahul', 'Priya', 'Aman', 'Karan', 'Neha', 'Vikas']

# Per-person message count (media + deleted count — they DID send something)
msg_count = {}
for m in messages:
    name = m['sender']
    msg_count[name] = msg_count.get(name, 0) + 1

# Date range
first_dt = datetime.strptime(messages[0]['timestamp'], '%d/%m/%y, %H:%M')
last_dt  = datetime.strptime(messages[-1]['timestamp'], '%d/%m/%y, %H:%M')
total_days_range = (last_dt.date() - first_dt.date()).days + 1
total_msgs = len(messages)

# Sorted per-person list
sorted_members = sorted(msg_count.items(), key=lambda x: x[1], reverse=True)
max_count = sorted_members[0][1]

print("=" * 62)
print(f"{'GROUPDNA REPORT — "Hostel Bois 4ever"':^62}")
print(f"{str(total_days_range) + ' days  •  ' + str(total_msgs) + ' messages  •  ' + str(len(msg_count)) + ' members':^62}")
print("=" * 62)
print(f"  Period         : {first_dt.strftime('%d %B %Y')} to {last_dt.strftime('%d %B %Y')} ({total_days_range} days)")
print(f"  Total messages : {total_msgs:,}")
print(f"  Participants   : {len(msg_count)}")
print()
print(f"  MESSAGES PER PERSON")
for name, count in sorted_members:
    bar_len = int((count / max_count) * 20)
    bar = '█' * bar_len if bar_len > 0 else '.'
    pct = count / total_msgs * 100
    print(f"  {name:<8}  {bar:<21} {count:>4} ({pct:>4.1f}%)")


## Feature 3 — Most Active Day and Hour

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 3: MOST ACTIVE DAY AND HOUR
# Uses: dict for counts, max() with key=, string parsing
# ─────────────────────────────────────────────────────────────────────────────

day_counts  = {}
hour_counts = {}

for m in messages:
    ts   = m['timestamp']                       # e.g. '01/04/24, 01:17'
    date = ts.split(',')[0]                     # '01/04/24'
    hour = int(ts.split(',')[1].strip()[:2])    # 1

    day_counts[date]  = day_counts.get(date, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1

busiest_day_key   = max(day_counts,  key=lambda x: day_counts[x])
busiest_day_count = day_counts[busiest_day_key]
busiest_day_dt    = datetime.strptime(busiest_day_key, '%d/%m/%y')

busiest_hour      = max(hour_counts, key=lambda x: hour_counts[x])
busiest_hour_avg  = hour_counts[busiest_hour] / total_days_range

print(f"  Busiest day    : {busiest_day_dt.strftime('%d %B %Y')} ({busiest_day_count} messages)")
print(f"  Busiest hour   : {busiest_hour:02d}:00 – {busiest_hour+1:02d}:00 (avg {busiest_hour_avg:.0f} msgs/day)")


## Feature 4 — Activity Heatmap (NumPy)
Builds a 6×24 NumPy matrix (person × hour) and renders it as a terminal heatmap using Unicode block characters.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 4: NUMPY ACTIVITY HEATMAP
# Builds a 6x24 matrix: rows = participants, cols = hours 0–23
# Each cell = total messages that person sent during that hour across all days
# Uses: np.zeros, integer indexing, sum, max, slicing
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np

person_index = {name: i for i, name in enumerate(PARTICIPANTS)}

# Build the matrix — pure NumPy, no pandas
heatmap = np.zeros((6, 24), dtype=int)

for m in messages:
    if m['sender'] in person_index:
        hour = int(m['timestamp'].split(',')[1].strip()[:2])
        heatmap[person_index[m['sender']], hour] += 1

# Shading levels based on each cell relative to that person's row maximum
SHADING = ['. ', '░ ', '▒ ', '█ ']

print("  ACTIVITY HEATMAP (messages by hour of day)")
print(f"  {'':9}  00 03 06 09 12 15 18 21")

for i, name in enumerate(PARTICIPANTS):
    row     = heatmap[i]
    row_max = row.max()
    row_str = ''
    for h in range(0, 24, 3):
        val = row[h]
        if row_max == 0:
            level = 0
        else:
            ratio = val / row_max
            if   ratio == 0:      level = 0
            elif ratio <= 0.25:   level = 0
            elif ratio <= 0.50:   level = 1
            elif ratio <= 0.75:   level = 2
            else:                 level = 3
        row_str += SHADING[level]

    # Annotate Aman as the night owl
    night_msgs = int(heatmap[i, 23]) + int(heatmap[i, 0:5].sum())
    total_p    = int(heatmap[i].sum())
    pct_note   = f'  ← {night_msgs/total_p*100:.0f}% at night' if name == 'Aman' else ''
    print(f"  {name:<9}  {row_str}{pct_note}")

print()
print(f"  [NumPy check] Matrix shape: {heatmap.shape}  |  Total cells: {heatmap.size}")
print(f"  [NumPy check] Grand total messages in matrix: {heatmap.sum()}")


## Feature 5 — Top Words
Word frequency analysis with custom stop words and bar-chart visualization.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 5: TOP WORDS
# Tokenizes every real message, strips punctuation, lower-cases, filters stop words
# Uses: loops, str.split(), str.lower(), str.strip(), dict, sorted()
# NOTE: No collections.Counter used — custom dict counter only
# ─────────────────────────────────────────────────────────────────────────────

STOP_WORDS = {
    'i','is','the','a','and','or','to','of','in','on','for','it','my','me',
    'we','you','he','she','they','this','that','was','are','be','but','so',
    'at','do','go','up','if','not','yes','no','ok','okay','its','with','just',
    'have','your','him','her','our','been','also','from','an','as','by','what',
    'how','when','where','who','will','would','about','get','got','all','one',
    'can','more','like','know','am','his','which','had','has','were','their',
    'there','then','them','into','out','over','after','before','some','any',
    'did','could','should','very','too','even','back','day','time','said',
    'told','came','went','come','going','every','really','actually','because',
    'only','much','started','ended','happened','since','while','being','whole',
    'completely','always','never','still','well','way','little','each','other',
    'another','most','tell','take','want','left','right','us','than','these',
    'those','such','both','during','through','having','doing','making','getting'
}

# Group-wide word counts
word_counts = {}

# Per-person word counts for bonus section
person_word_counts = {p: {} for p in PARTICIPANTS}

for m in messages:
    if m['is_media'] or m['is_deleted']:
        continue
    words = m['text'].lower().split()
    for w in words:
        # Strip punctuation manually (no regex allowed)
        clean = ''
        for ch in w:
            if ch.isalpha():
                clean += ch
        if len(clean) <= 1 or clean in STOP_WORDS:
            continue
        # Group counter
        word_counts[clean] = word_counts.get(clean, 0) + 1
        # Per-person counter
        if m['sender'] in person_word_counts:
            pwc = person_word_counts[m['sender']]
            pwc[clean] = pwc.get(clean, 0) + 1

top_words  = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]
top_count  = top_words[0][1]

print("  THIS GROUP'S FAVOURITE WORDS")
print()
for word, count in top_words:
    bar_len = int((count / top_count) * 20)
    bar = '█' * bar_len
    print(f"  {word:<14}  {bar:<21}  {count}")

print()
print("  TOP 3 WORDS PER PERSON")
print()
for p in PARTICIPANTS:
    top3 = sorted(person_word_counts[p].items(), key=lambda x: x[1], reverse=True)[:3]
    words_str = '  |  '.join(f"{w} ({c})" for w, c in top3)
    print(f"  {p:<8}:  {words_str}")


## Feature 6 — Response Speed & Silent Streaks

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 6: RESPONSE SPEED & SILENT STREAKS
# Uses: datetime.strptime, timedelta, loops, dicts, streak counting
# ─────────────────────────────────────────────────────────────────────────────
from datetime import datetime, timedelta


def parse_ts(ts_str):
    """Convert '01/04/24, 01:17' to a datetime object."""
    return datetime.strptime(ts_str, '%d/%m/%y, %H:%M')


# ── 6a: Average response time ─────────────────────────────────────────────────
# For each message: if the sender is different from the previous sender,
# the gap is a potential response time. We cap at 6 hours (21600 secs)
# to exclude overnight gaps that are not real replies.

response_gaps = {p: [] for p in PARTICIPANTS}

for i in range(1, len(messages)):
    curr = messages[i]
    prev = messages[i - 1]
    if curr['sender'] == prev['sender']:
        continue                            # same person, not a reply
    gap_secs = (parse_ts(curr['timestamp']) - parse_ts(prev['timestamp'])).total_seconds()
    if 0 < gap_secs < 21600:               # cap at 6 hours
        response_gaps[curr['sender']].append(gap_secs)

avg_response = {}
for p in PARTICIPANTS:
    gaps = response_gaps[p]
    avg_response[p] = sum(gaps) / len(gaps) if gaps else float('inf')


def fmt_time(secs):
    if secs == float('inf'):
        return 'N/A'
    if secs < 3600:
        return f"{secs / 60:.1f} minutes"
    return f"{secs / 3600:.1f} hours"


valid_responders = [p for p in PARTICIPANTS if avg_response[p] != float('inf')]
fastest = min(valid_responders, key=lambda x: avg_response[x])
slowest = max(valid_responders, key=lambda x: avg_response[x])

print("  RESPONSE PATTERNS")
print(f"  Fastest replier : {fastest} (avg {fmt_time(avg_response[fastest])})")
print(f"  Slowest replier : {slowest} (avg {fmt_time(avg_response[slowest])})")
print()
for p in PARTICIPANTS:
    print(f"  {p:<8}  avg response: {fmt_time(avg_response[p])}")


# ── 6b: Longest silent streaks ────────────────────────────────────────────────
# Build set of active days per person, then find longest gap in the 60-day window.

first_day = parse_ts(messages[0]['timestamp']).date()
last_day  = parse_ts(messages[-1]['timestamp']).date()

all_days = []
d = first_day
while d <= last_day:
    all_days.append(d)
    d += timedelta(days=1)

active_days_set = {p: set() for p in PARTICIPANTS}
for m in messages:
    if m['sender'] in active_days_set:
        active_days_set[m['sender']].add(parse_ts(m['timestamp']).date())

longest_streak  = {}
streak_start_dt = {}

for p in PARTICIPANTS:
    active    = active_days_set[p]
    max_s     = 0
    curr_s    = 0
    curr_start = None
    best_start = None

    for day in all_days:
        if day not in active:
            if curr_s == 0:
                curr_start = day
            curr_s += 1
            if curr_s > max_s:
                max_s      = curr_s
                best_start = curr_start
        else:
            curr_s = 0

    longest_streak[p]  = max_s
    streak_start_dt[p] = best_start

print()
print("  LONGEST SILENT STREAKS (consecutive days with zero messages)")
sorted_streaks = sorted(PARTICIPANTS, key=lambda x: longest_streak[x], reverse=True)
for p in sorted_streaks:
    s = longest_streak[p]
    if s == 0:
        print(f"  {p:<8} : 0 days (never went silent)")
    else:
        end_date = streak_start_dt[p] + timedelta(days=s - 1)
        print(f"  {p:<8} : {s} days ({streak_start_dt[p].strftime('%d %b')} to {end_date.strftime('%d %b')})")


## Feature 7 — Personality Archetype Detection
8 archetypes with explicit detection rules. Assignment is **exclusive** — each archetype goes to only one person (highest scorer wins). Tie-break rule: higher raw score takes priority; if equal, earlier in the archetype priority list wins.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 7: PERSONALITY ARCHETYPE DETECTION
# 8 archetypes + 1 bonus, one detection function each.
# Assignment is EXCLUSIVE: scores are normalized to 0-1, then the highest
# normalized scorer for each archetype wins.
# Tie-break rule (documented): primary archetypes (1-8) take priority over
# the bonus archetype; within equal normalized scores, lower archetype
# priority index wins (i.e. earlier in the ARCH_ORDER list).
# Uses: functions, dicts, conditionals, heatmap from Feature 4, string ops
# ─────────────────────────────────────────────────────────────────────────────

# ── Detection functions ───────────────────────────────────────────────────────

def score_spammer(person):
    """Avg length of consecutive message bursts without anyone else speaking."""
    bursts  = []
    curr    = 0
    for m in messages:
        if m['sender'] == person:
            curr += 1
        else:
            if curr > 0:
                bursts.append(curr)
            curr = 0
    if curr > 0:
        bursts.append(curr)
    return sum(bursts) / len(bursts) if bursts else 0


CARING_KEYWORDS = [
    'okay', 'safe', 'eat', 'sleep', 'take care', 'are you',
    'please', 'reminder', 'drink water', "don't forget",
    'dont forget', 'doing okay', 'everyone okay'
]

def score_group_mom(person):
    """Total count of caring keywords across all this person's messages."""
    count = 0
    for m in messages:
        if m['sender'] == person and not m['is_media'] and not m['is_deleted']:
            txt = m['text'].lower()
            for kw in CARING_KEYWORDS:
                count += txt.count(kw)
    return count


def score_night_owl(person):
    """Fraction of messages sent between 23:00 and 04:59."""
    idx   = person_index[person]
    row   = heatmap[idx]
    night = int(row[23]) + int(row[0:5].sum())
    total = int(row.sum())
    return night / total if total > 0 else 0


def score_storyteller(person):
    """Average words per message."""
    wc = [len(m['text'].split()) for m in messages
          if m['sender'] == person and not m['is_media'] and not m['is_deleted']]
    return sum(wc) / len(wc) if wc else 0


def score_drama_queen(person):
    """Fraction of messages that are fully uppercase OR have 2+ exclamation marks."""
    total = drama = 0
    for m in messages:
        if m['sender'] == person and not m['is_media'] and not m['is_deleted']:
            txt = m['text']
            if len(txt) < 3:
                continue
            total += 1
            if txt.isupper() or txt.count('!') >= 2:
                drama += 1
    return drama / total if total > 0 else 0


def score_ghost(person):
    """Fraction of the 60-day period the person was completely silent."""
    return (len(all_days) - len(active_days_set[person])) / len(all_days)


COMEDIAN_KEYWORDS = ['lol', 'lmao', 'haha', 'rofl', 'lmfao', 'hahaha', 'hahahaha']

def score_comedian(person):
    """Fraction of messages containing a laugh keyword."""
    total = comedy = 0
    for m in messages:
        if m['sender'] == person and not m['is_media'] and not m['is_deleted']:
            total += 1
            txt = m['text'].lower()
            for kw in COMEDIAN_KEYWORDS:
                if kw in txt:
                    comedy += 1
                    break
    return comedy / total if total > 0 else 0


def score_question_master(person):
    """Fraction of messages ending with a question mark."""
    total = q_count = 0
    for m in messages:
        if m['sender'] == person and not m['is_media'] and not m['is_deleted']:
            total += 1
            if m['text'].strip().endswith('?'):
                q_count += 1
    return q_count / total if total > 0 else 0


# ── BONUS ARCHETYPE: THE LATE-NIGHT PHILOSOPHER ───────────────────────────────
# Inspired by Indian college hostel life: that one person who sends deep thoughts
# about life and existence after midnight when everyone else is asleep.
# Detection: count of philosophical keywords in messages sent after 23:00 or before 05:00.

PHILOSOPHER_KEYWORDS = [
    'life', 'time', 'meaning', 'exist', 'wonder', 'think', 'feel',
    'reality', 'world', 'purpose', 'future', 'past', 'dream', 'truth'
]

def score_late_night_philosopher(person):
    """Count of philosophical keywords in late-night messages (23:00–04:59)."""
    count = 0
    for m in messages:
        if m['sender'] == person and not m['is_media'] and not m['is_deleted']:
            hour = int(m['timestamp'].split(',')[1].strip()[:2])
            if hour >= 23 or hour <= 4:
                txt = m['text'].lower()
                for kw in PHILOSOPHER_KEYWORDS:
                    count += txt.count(kw)
    return count


# ── Collect raw scores ────────────────────────────────────────────────────────

# Archetype order — primary 8 come before the bonus (tie-break priority)
ARCH_ORDER = [
    'THE SPAMMER',
    'THE GROUP MOM',
    'THE NIGHT OWL',
    'THE STORYTELLER',
    'THE DRAMA QUEEN',
    'THE GHOST',
    'THE COMEDIAN',
    'THE QUESTION MASTER',
    'THE LATE-NIGHT PHILOSOPHER',   # BONUS — lower priority in tie-breaks
]

SCORE_FUNCTIONS = {
    'THE SPAMMER'               : score_spammer,
    'THE GROUP MOM'             : score_group_mom,
    'THE NIGHT OWL'             : score_night_owl,
    'THE STORYTELLER'           : score_storyteller,
    'THE DRAMA QUEEN'           : score_drama_queen,
    'THE GHOST'                 : score_ghost,
    'THE COMEDIAN'              : score_comedian,
    'THE QUESTION MASTER'       : score_question_master,
    'THE LATE-NIGHT PHILOSOPHER': score_late_night_philosopher,
}

raw_scores = {p: {arch: SCORE_FUNCTIONS[arch](p) for arch in ARCH_ORDER}
              for p in PARTICIPANTS}

# ── Normalize scores to 0–1 per archetype ────────────────────────────────────
# This ensures scores from different units (fractions vs raw counts vs averages)
# are comparable when doing exclusive assignment.

norm_scores = {p: {} for p in PARTICIPANTS}
for arch in ARCH_ORDER:
    arch_max = max(raw_scores[p][arch] for p in PARTICIPANTS)
    if arch_max == 0:
        arch_max = 1   # avoid division by zero
    for p in PARTICIPANTS:
        norm_scores[p][arch] = raw_scores[p][arch] / arch_max

# ── Exclusive assignment ──────────────────────────────────────────────────────
# Build a flat list of (norm_score, -arch_priority_index, person, archetype).
# Sorting descending by norm_score then by -arch_priority_index means:
#   1. Higher normalized score wins.
#   2. Ties broken by archetype priority (primary archetypes beat bonus).
# Each person gets exactly one archetype; each archetype assigned to one person.

flat = []
for p in PARTICIPANTS:
    for arch in ARCH_ORDER:
        priority = -ARCH_ORDER.index(arch)   # higher (less negative) = higher priority
        flat.append((norm_scores[p][arch], priority, p, arch))

flat.sort(key=lambda x: (x[0], x[1]), reverse=True)

assigned        = {}
used_archetypes = set()

for score_val, priority, person, arch in flat:
    if person not in assigned and arch not in used_archetypes:
        assigned[person] = (arch, raw_scores[person][arch])
        used_archetypes.add(arch)


# ── Helper for detail string ──────────────────────────────────────────────────

def archetype_detail(arch, raw_sc, person):
    if arch == 'THE SPAMMER':
        return f"avg {raw_sc:.1f} msgs in a row"
    elif arch == 'THE GROUP MOM':
        return f"caring keyword score: {int(raw_sc)}"
    elif arch == 'THE NIGHT OWL':
        return f"{raw_sc*100:.1f}% msgs between 23h–04h"
    elif arch == 'THE STORYTELLER':
        return f"avg {raw_sc:.1f} words per msg"
    elif arch == 'THE DRAMA QUEEN':
        return f"{raw_sc*100:.1f}% ALL-CAPS or 2+ exclamation messages"
    elif arch == 'THE GHOST':
        silent = int(raw_sc * len(all_days))
        return f"silent on {silent} of {len(all_days)} days"
    elif arch == 'THE COMEDIAN':
        return f"{raw_sc*100:.1f}% of msgs have laugh keywords"
    elif arch == 'THE QUESTION MASTER':
        return f"{raw_sc*100:.1f}% messages end with ?"
    elif arch == 'THE LATE-NIGHT PHILOSOPHER':
        return f"used {int(raw_sc)} philosophical keywords after 11 PM"
    return str(raw_sc)


# ── Print results ─────────────────────────────────────────────────────────────

print("  PERSONALITY ARCHETYPES")
print()
for p in PARTICIPANTS:
    arch, sc = assigned[p]
    detail   = archetype_detail(arch, sc, p)
    print(f"  {p:<8}  →  {arch:<32}  ({detail})")

print()
print("  BONUS ARCHETYPE SCORES (THE LATE-NIGHT PHILOSOPHER)")
for p in PARTICIPANTS:
    sc = raw_scores[p]['THE LATE-NIGHT PHILOSOPHER']
    print(f"  {p:<8}  :  {int(sc)} late-night philosophical keywords")


## Feature 8 — The Final Report
All sections assembled into one beautiful, screenshot-ready printed report.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 8: THE FINAL REPORT
# Combines all features into one formatted, screenshot-ready output.
# Uses: f-strings with width specifiers, string multiplication, print()
# ─────────────────────────────────────────────────────────────────────────────

W = 64   # report width

def divider(char='='):
    print(char * W)

def section(title):
    print()
    print(f"  ── {title} " + "─" * (W - len(title) - 6))

def blank():
    print()


divider()
print(f"{'GROUPDNA REPORT — "Hostel Bois 4ever"':^{W}}")
print(f"{str(total_days_range) + ' days  •  ' + str(total_msgs) + ' messages  •  ' + str(len(msg_count)) + ' members':^{W}}")
divider()

# ── Section: Overview ─────────────────────────────────────────────────────────
blank()
print(f"  Period         : {first_dt.strftime('%d %B %Y')} to {last_dt.strftime('%d %B %Y')} ({total_days_range} days)")
print(f"  Busiest day    : {busiest_day_dt.strftime('%d %B %Y')} ({busiest_day_count} messages)")
print(f"  Busiest hour   : {busiest_hour:02d}:00 – {busiest_hour+1:02d}:00 (avg {busiest_hour_avg:.0f} msgs/day)")
print(f"  System msgs    : {sys_skip}  |  Media-omitted: {media_skip}  |  Deleted: {del_skip}")

# ── Section: Messages per person ──────────────────────────────────────────────
section("MESSAGES PER PERSON")
blank()
for name, count in sorted_members:
    bar_len = int((count / max_count) * 20)
    bar = '█' * bar_len if bar_len > 0 else '.'
    pct = count / total_msgs * 100
    print(f"  {name:<8}  {bar:<21} {count:>4} ({pct:>4.1f}%)")

# ── Section: Heatmap ──────────────────────────────────────────────────────────
section("ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
blank()
print(f"  {'':9}  00 03 06 09 12 15 18 21")
for i, name in enumerate(PARTICIPANTS):
    row     = heatmap[i]
    row_max = row.max()
    row_str = ''
    for h in range(0, 24, 3):
        val = row[h]
        if row_max == 0:
            level = 0
        else:
            ratio = val / row_max
            if   ratio == 0:      level = 0
            elif ratio <= 0.25:   level = 0
            elif ratio <= 0.50:   level = 1
            elif ratio <= 0.75:   level = 2
            else:                 level = 3
        row_str += SHADING[level]
    night_note = '  ← NIGHT OWL' if name == 'Aman' else ''
    print(f"  {name:<9}  {row_str}{night_note}")

# ── Section: Top words ────────────────────────────────────────────────────────
section("THIS GROUP'S FAVOURITE WORDS")
blank()
for word, count in top_words:
    bar_len = int((count / top_count) * 20)
    bar = '█' * bar_len
    print(f"  {word:<14}  {bar:<21}  {count}")

# ── Section: Response patterns ────────────────────────────────────────────────
section("RESPONSE PATTERNS")
blank()
print(f"  Fastest replier : {fastest} (avg {fmt_time(avg_response[fastest])})")
print(f"  Slowest replier : {slowest} (avg {fmt_time(avg_response[slowest])})")

# ── Section: Silent streaks ───────────────────────────────────────────────────
section("LONGEST SILENT STREAKS")
blank()
for p in sorted_streaks:
    s = longest_streak[p]
    if s == 0:
        print(f"  {p:<8} : 0 days (never went silent)")
    else:
        end_date = streak_start_dt[p] + timedelta(days=s - 1)
        print(f"  {p:<8} : {s} days ({streak_start_dt[p].strftime('%d %b')} to {end_date.strftime('%d %b')})")

# ── Section: Archetypes ───────────────────────────────────────────────────────
section("PERSONALITY ARCHETYPES")
blank()
for p in PARTICIPANTS:
    arch, sc = assigned[p]
    detail   = archetype_detail(arch, sc, p)
    print(f"  {p:<8}  →  {arch:<30}  ({detail})")

# ── Footer ────────────────────────────────────────────────────────────────────
blank()
divider()
print(f"{'Generated by GroupDNA  •  Built with Python + NumPy':^{W}}")
print(f"{'No pandas. No matplotlib. No regex. Pure fundamentals.':^{W}}")
divider()


## Reflection

**What was the hardest part?**
The hardest part was getting the parser to handle all 5 edge cases cleanly — especially making sure system messages were filtered properly without accidentally dropping real messages. The multi-line message logic required careful thought because any line that didn't start with a date needed to be appended to the previous message's text.

**What would I do differently?**
I would spend more time on the heatmap rendering early — the visual output is what makes the report shareable, and I underestimated how much the formatting matters until I saw the final output side-by-side with the brief's reference.

**Key learnings:**
- Real-world data is always messier than it looks. The WhatsApp export had system messages, deleted stubs, media placeholders, and continuation lines all mixed together.
- NumPy is powerful even for simple integer counting — the 6×24 matrix made the night-owl detection a one-liner (`row[23] + row[0:5].sum()`).
- Output formatting is not optional. A polished terminal report looks completely different from a raw `print()` dump.

**What archetype did I get on my own chat?** *(run it yourself and fill this in!)*

---
*Built using: Python fundamentals + NumPy only. No pandas, no matplotlib, no regex, no collections.Counter.*
